# Notebook 17: Advanced Biophysical Modeling

## Bridging Neural Networks and Biological Reality

**Author**: NeuroS Team  
**Date**: 2025-11-25  
**Duration**: ~25 minutes

---

## Overview

This notebook demonstrates the advanced biophysical modeling capabilities of `neuros-mechint`, enabling biologically realistic simulations for foundation model testing:

### Core Topics:

1. **Ion Channel Dynamics** - Hodgkin-Huxley formalism for voltage-gated conductances
2. **Multi-Compartment Neurons** - Cable theory, dendritic integration, and spatial structure
3. **Synaptic Plasticity** - STDP, short-term plasticity, and homeostatic mechanisms
4. **Metabolic Constraints** - ATP dynamics, energy budgets, and metabolic realism
5. **Foundation Model Testing** - Applying biophysical constraints to AI predictions

### Why This Matters:

Foundation models often produce representations that are mathematically elegant but biologically implausible. By incorporating biophysical constraints—ion channel dynamics, metabolic limits, realistic plasticity rules—we can:

- **Test biological plausibility** of model representations
- **Constrain search spaces** for mechanistic hypotheses
- **Bridge scales** from molecular (channels) to network (circuits)
- **Guide architecture design** using principles from neuroscience

### Key References:

- **Hodgkin & Huxley (1952)**: Quantitative description of membrane current
- **Rall (1959)**: Dendritic cable theory
- **Bi & Poo (1998)**: Synaptic modifications by correlated activity (STDP)
- **Attwell & Laughlin (2001)**: Energy budget of signaling in the brain

---

In [ ]:
# Core imports
import numpy as np
import torch
import torch.nn as nn
from typing import List, Tuple, Dict
import warnings
warnings.filterwarnings('ignore')

# Bokeh for interactive visualizations
from bokeh.plotting import figure, show, output_notebook
from bokeh.layouts import gridplot, column, row
from bokeh.models import HoverTool, Span, Label, ColorBar, LinearColorMapper
from bokeh.palettes import Viridis256, Turbo256, RdYlBu11

# NeuroS-MechInt biophysical components
from neuros_mechint.biophysical import (
    MultiCompartmentNeuron, PrefabNeurons,
    STDP, STDPParameters, ShortTermPlasticity,
    ATPDynamics, MetabolicConstraint
)

# Enable Bokeh in Jupyter
output_notebook()

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("✓ All imports successful")
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

---

## Part 1: Ion Channel Dynamics

### The Hodgkin-Huxley Model

The Hodgkin-Huxley (HH) model describes the ionic mechanisms underlying the action potential. The membrane potential $V$ evolves according to:

$$
C_m \frac{dV}{dt} = I_{\text{ext}} - I_{\text{Na}} - I_{\text{K}} - I_{\text{L}}
$$

where:
- $C_m$ is membrane capacitance (typically 1 μF/cm²)
- $I_{\text{ext}}$ is external current injection
- $I_{\text{Na}} = g_{\text{Na}} m^3 h (V - E_{\text{Na}})$ is sodium current
- $I_{\text{K}} = g_{\text{K}} n^4 (V - E_{\text{K}})$ is potassium current  
- $I_{\text{L}} = g_{\text{L}} (V - E_{\text{L}})$ is leak current

**Gating variables** ($m$, $h$, $n$) follow first-order kinetics:

$$
\frac{dx}{dt} = \alpha_x(V)(1 - x) - \beta_x(V) x
$$

where $x \in \{m, h, n\}$ and $\alpha$, $\beta$ are voltage-dependent rate constants.

### Interactive Exploration

Let's simulate ion channel dynamics and visualize their voltage-dependent properties.

In [ ]:
# Create Hodgkin-Huxley neuron model
class HodgkinHuxleyNeuron:
    """Complete Hodgkin-Huxley neuron model."""
    
    def __init__(self, device='cpu'):
        self.device = device
        
        # Membrane capacitance (μF/cm²)
        self.C_m = 1.0
        
        # Maximum conductances (mS/cm²)
        self.g_Na = 120.0
        self.g_K = 36.0
        self.g_L = 0.3
        
        # Reversal potentials (mV)
        self.E_Na = 50.0
        self.E_K = -77.0
        self.E_L = -54.387
        
        # Initialize state
        self.V = -65.0  # Resting potential
        self.m = 0.05
        self.h = 0.6
        self.n = 0.32
        
    def alpha_m(self, V):
        return 0.1 * (V + 40.0) / (1.0 - np.exp(-(V + 40.0) / 10.0))
    
    def beta_m(self, V):
        return 4.0 * np.exp(-(V + 65.0) / 18.0)
    
    def alpha_h(self, V):
        return 0.07 * np.exp(-(V + 65.0) / 20.0)
    
    def beta_h(self, V):
        return 1.0 / (1.0 + np.exp(-(V + 35.0) / 10.0))
    
    def alpha_n(self, V):
        return 0.01 * (V + 55.0) / (1.0 - np.exp(-(V + 55.0) / 10.0))
    
    def beta_n(self, V):
        return 0.125 * np.exp(-(V + 65.0) / 80.0)
    
    def step(self, I_ext, dt=0.01):
        """Advance simulation by dt (ms)."""
        # Compute currents
        I_Na = self.g_Na * (self.m ** 3) * self.h * (self.V - self.E_Na)
        I_K = self.g_K * (self.n ** 4) * (self.V - self.E_K)
        I_L = self.g_L * (self.V - self.E_L)
        
        # Update voltage
        dV = (I_ext - I_Na - I_K - I_L) / self.C_m
        self.V += dV * dt
        
        # Update gating variables
        dm = self.alpha_m(self.V) * (1 - self.m) - self.beta_m(self.V) * self.m
        dh = self.alpha_h(self.V) * (1 - self.h) - self.beta_h(self.V) * self.h
        dn = self.alpha_n(self.V) * (1 - self.n) - self.beta_n(self.V) * self.n
        
        self.m += dm * dt
        self.h += dh * dt
        self.n += dn * dt
        
        return self.V, I_Na, I_K, I_L

print("✓ Hodgkin-Huxley neuron model defined")

In [ ]:
# Simulate action potential
neuron = HodgkinHuxleyNeuron()

# Simulation parameters
dt = 0.01  # ms
duration = 50  # ms
n_steps = int(duration / dt)

# Current injection protocol: brief pulse to trigger spike
I_ext = np.zeros(n_steps)
I_ext[500:700] = 10.0  # 10 μA/cm² for 2 ms

# Storage
time = np.arange(n_steps) * dt
V_trace = np.zeros(n_steps)
I_Na_trace = np.zeros(n_steps)
I_K_trace = np.zeros(n_steps)
I_L_trace = np.zeros(n_steps)
m_trace = np.zeros(n_steps)
h_trace = np.zeros(n_steps)
n_trace = np.zeros(n_steps)

# Run simulation
for i in range(n_steps):
    V, I_Na, I_K, I_L = neuron.step(I_ext[i], dt)
    V_trace[i] = V
    I_Na_trace[i] = I_Na
    I_K_trace[i] = I_K
    I_L_trace[i] = I_L
    m_trace[i] = neuron.m
    h_trace[i] = neuron.h
    n_trace[i] = neuron.n

print(f"✓ Simulated {duration} ms")
print(f"✓ Peak voltage: {V_trace.max():.1f} mV")
print(f"✓ Peak Na+ current: {I_Na_trace.max():.1f} μA/cm²")

In [ ]:
# Interactive Bokeh visualization of action potential

# Panel 1: Membrane voltage
p1 = figure(
    title='Membrane Voltage',
    width=900,
    height=250,
    x_axis_label='Time (ms)',
    y_axis_label='Voltage (mV)'
)
p1.line(time, V_trace, line_width=2, color='navy', legend_label='V(t)')
hover1 = HoverTool(tooltips=[('Time', '@x{0.0} ms'), ('Voltage', '@y{0.0} mV')])
p1.add_tools(hover1)
p1.legend.location = 'top_right'

# Panel 2: Ionic currents
p2 = figure(
    title='Ionic Currents',
    width=900,
    height=250,
    x_axis_label='Time (ms)',
    y_axis_label='Current (μA/cm²)',
    x_range=p1.x_range
)
p2.line(time, I_Na_trace, line_width=2, color='red', legend_label='I_Na', alpha=0.8)
p2.line(time, I_K_trace, line_width=2, color='blue', legend_label='I_K', alpha=0.8)
p2.line(time, I_L_trace, line_width=2, color='green', legend_label='I_L', alpha=0.6)
hover2 = HoverTool(tooltips=[('Time', '@x{0.0} ms'), ('Current', '@y{0.0} μA/cm²')])
p2.add_tools(hover2)
p2.legend.location = 'top_right'

# Panel 3: Gating variables
p3 = figure(
    title='Gating Variables',
    width=900,
    height=250,
    x_axis_label='Time (ms)',
    y_axis_label='Activation',
    x_range=p1.x_range,
    y_range=(0, 1)
)
p3.line(time, m_trace, line_width=2, color='red', legend_label='m (Na+ activation)', alpha=0.8)
p3.line(time, h_trace, line_width=2, color='orange', legend_label='h (Na+ inactivation)', alpha=0.8)
p3.line(time, n_trace, line_width=2, color='blue', legend_label='n (K+ activation)', alpha=0.8)
hover3 = HoverTool(tooltips=[('Time', '@x{0.0} ms'), ('Value', '@y{0.000}')])
p3.add_tools(hover3)
p3.legend.location = 'center_right'

# Panel 4: Current injection
p4 = figure(
    title='Current Injection',
    width=900,
    height=200,
    x_axis_label='Time (ms)',
    y_axis_label='I_ext (μA/cm²)',
    x_range=p1.x_range
)
p4.line(time, I_ext, line_width=2, color='black')
hover4 = HoverTool(tooltips=[('Time', '@x{0.0} ms'), ('Current', '@y{0.0} μA/cm²')])
p4.add_tools(hover4)

# Combine panels
grid = gridplot([[p1], [p2], [p3], [p4]])
show(grid)

print("\n📊 Interactive Action Potential Visualization")
print("   Hover over traces to see detailed values")
print("   Zoom and pan to explore specific phases")

---

## Part 2: Multi-Compartment Neurons

### Cable Theory and Dendritic Integration

Real neurons have complex morphologies. **Cable theory** (Rall, 1959) describes how electrical signals propagate along dendrites. The voltage $V(x, t)$ at position $x$ along a cable evolves as:

$$
\tau \frac{\partial V}{\partial t} = \lambda^2 \frac{\partial^2 V}{\partial x^2} - V
$$

where:
- $\tau = R_m C_m$ is membrane time constant
- $\lambda = \sqrt{R_m / R_i}$ is space constant (length scale)
- $R_m$ is membrane resistance, $R_i$ is axial resistance

**Multi-compartment models** discretize the cable equation, treating each segment as a point neuron coupled by axial resistances.

### Pyramidal Neuron Simulation

Let's simulate a cortical pyramidal cell with soma, basal dendrites, and apical dendrite.

In [ ]:
# Create multi-compartment pyramidal neuron - FIXED API
pyramidal = PrefabNeurons.pyramidal_cell(include_channels=True, dt=0.05)

print(f"✓ Pyramidal neuron created")
print(f"  Compartments: {len(pyramidal.compartments)}")
for i, comp in enumerate(pyramidal.compartments):
    comp_names = ['Soma', 'Basal Dendrite', 'Apical Dendrite']
    if i < len(comp_names):
        print(f"    - {comp_names[i]}: area = {comp.geometry.area:.1f} μm²")

In [ ]:
# Simulate with dendritic current injection - FIXED API
dt = 0.05  # ms
duration = 100  # ms
n_steps = int(duration / dt)
batch_size = 1
device = torch.device('cpu')

# Initialize all compartments first
for comp in pyramidal.compartments:
    if comp.V is None:
        comp.initialize(batch_size, device)

print(f"✓ Compartments initialized")

# Create synaptic currents for each compartment (mostly zero)
I_syn = [torch.zeros(batch_size, device=device) for _ in range(len(pyramidal.compartments))]

# Create stimulus current (inject into apical dendrite - compartment 2)
I_stim = [torch.zeros(batch_size, device=device) for _ in range(len(pyramidal.compartments))]

# Create time-varying current injection into apical dendrite
current_injection = torch.zeros(n_steps)
current_injection[400:800] = 200.0  # 200 pA for 20 ms

# Run simulation
voltages = []
for t in range(n_steps):
    # Update stimulus
    I_stim[2] = torch.tensor([current_injection[t]], device=device)  # Inject into apical dendrite
    
    # Update each compartment
    for i, comp in enumerate(pyramidal.compartments):
        comp.step(I_syn[i], I_stim[i], None)
    
    # Record all compartment voltages
    voltages.append([comp.V.item() for comp in pyramidal.compartments])

voltages = torch.tensor(voltages)  # Shape: (n_steps, n_compartments)
time_mc = np.arange(n_steps) * dt

print(f"\n✓ Simulation complete")
print(f"  Soma spikes: {((voltages[:, 0] > -20).diff(dim=0) == 1).sum().item()}")
print(f"  Peak soma voltage: {voltages[:, 0].max().item():.1f} mV")

In [ ]:
# Interactive visualization of multi-compartment neuron

# Panel 1: Current injection
p_curr = figure(
    title='Dendritic Current Injection',
    width=900,
    height=200,
    x_axis_label='Time (ms)',
    y_axis_label='Current (pA)'
)
p_curr.line(time_mc, current_injection.numpy(), line_width=2, color='black')
hover_curr = HoverTool(tooltips=[('Time', '@x{0.0} ms'), ('Current', '@y{0.0} pA')])
p_curr.add_tools(hover_curr)

# Panel 2: All compartment voltages
p_all = figure(
    title='Voltage in All Compartments',
    width=900,
    height=350,
    x_axis_label='Time (ms)',
    y_axis_label='Voltage (mV)',
    x_range=p_curr.x_range
)
colors_comp = ['red', 'blue', 'green']
comp_names = ['Soma', 'Basal Dendrite', 'Apical Dendrite']
for i in range(len(pyramidal.compartments)):
    color = colors_comp[i % len(colors_comp)]
    name = comp_names[i] if i < len(comp_names) else f'Comp {i}'
    p_all.line(time_mc, voltages[:, i].numpy(), line_width=2, 
               color=color, legend_label=name, alpha=0.8)
hover_all = HoverTool(tooltips=[('Time', '@x{0.0} ms'), ('Voltage', '@y{0.0} mV')])
p_all.add_tools(hover_all)
p_all.legend.location = 'top_right'
p_all.legend.click_policy = 'hide'

# Panel 3: Soma detail
p_soma = figure(
    title='Soma Response (Action Potentials)',
    width=900,
    height=300,
    x_axis_label='Time (ms)',
    y_axis_label='Soma Voltage (mV)',
    x_range=p_curr.x_range
)
p_soma.line(time_mc, voltages[:, 0].numpy(), line_width=2, color='darkred')
# Add spike threshold line
spike_line = Span(location=-20, dimension='width', line_color='red', 
                  line_dash='dashed', line_width=2)
p_soma.add_layout(spike_line)
hover_soma = HoverTool(tooltips=[('Time', '@x{0.0} ms'), ('Voltage', '@y{0.0} mV')])
p_soma.add_tools(hover_soma)

# Combine
grid_mc = gridplot([[p_curr], [p_all], [p_soma]])
show(grid_mc)

# Count and display spikes
soma_voltage = voltages[:, 0].numpy()
spikes = np.where((soma_voltage[:-1] < -20) & (soma_voltage[1:] >= -20))[0]
print(f"\n📊 Multi-Compartment Neuron Simulation")
print(f"   Total spikes: {len(spikes)}")
print(f"   Firing rate: {len(spikes) / (duration / 1000):.1f} Hz")
if len(spikes) > 1:
    isis = np.diff(spikes) * dt
    print(f"   Mean ISI: {isis.mean():.1f} ms")
    print(f"   CV of ISI: {isis.std() / isis.mean():.3f}")

---

## Part 3: Synaptic Plasticity

### Spike-Timing-Dependent Plasticity (STDP)

STDP is a Hebbian learning rule where synaptic strength changes depend on the relative timing of pre- and postsynaptic spikes:

$$
\Delta w = \begin{cases}
A_+ e^{-\Delta t / \tau_+} & \text{if } \Delta t > 0 \text{ (LTP)} \\
-A_- e^{\Delta t / \tau_-} & \text{if } \Delta t < 0 \text{ (LTD)}
\end{cases}
$$

where $\Delta t = t_{\text{post}} - t_{\text{pre}}$.

**Key insight**: Pre-before-post → LTP (potentiation), Post-before-pre → LTD (depression)

### Short-Term Plasticity (STP)

Synapses also exhibit dynamics on millisecond-to-second timescales through **facilitation** (enhancement) and **depression** (depletion).

In [ ]:
# STDP learning window - FIXED API
stdp_params = STDPParameters(
    tau_plus=20.0,
    tau_minus=20.0,
    A_plus=0.01,
    A_minus=0.012
)
stdp = STDP(params=stdp_params, dt=1.0)

delta_t_range = np.linspace(-100, 100, 400)  # ms
weight_changes = []

for delta_t in delta_t_range:
    if delta_t > 0:  # Pre before post → LTP
        dw = stdp_params.A_plus * np.exp(-delta_t / stdp_params.tau_plus)
    else:  # Post before pre → LTD
        dw = -stdp_params.A_minus * np.exp(delta_t / stdp_params.tau_minus)
    weight_changes.append(dw)

weight_changes = np.array(weight_changes)

print("✓ STDP learning window computed")
print(f"  Max LTP: {weight_changes.max():.4f}")
print(f"  Max LTD: {weight_changes.min():.4f}")

In [ ]:
# Interactive STDP window visualization

p_stdp = figure(
    title='STDP Learning Window',
    width=700,
    height=450,
    x_axis_label='Δt = t_post - t_pre (ms)',
    y_axis_label='Δw (weight change)'
)

# Split into LTP and LTD regions for coloring
ltp_mask = delta_t_range >= 0
ltd_mask = delta_t_range < 0

p_stdp.line(delta_t_range[ltp_mask], weight_changes[ltp_mask], 
            line_width=3, color='green', legend_label='LTP (potentiation)', alpha=0.8)
p_stdp.line(delta_t_range[ltd_mask], weight_changes[ltd_mask], 
            line_width=3, color='red', legend_label='LTD (depression)', alpha=0.8)

# Add reference lines
zero_h = Span(location=0, dimension='width', line_color='black', line_dash='dashed', line_width=1)
zero_v = Span(location=0, dimension='height', line_color='black', line_dash='dashed', line_width=1)
p_stdp.add_layout(zero_h)
p_stdp.add_layout(zero_v)

# Add annotations
ltp_label = Label(x=30, y=0.008, text='LTP\n(pre→post)', 
                  text_font_size='12pt', text_align='center', text_color='green')
ltd_label = Label(x=-30, y=-0.010, text='LTD\n(post→pre)', 
                  text_font_size='12pt', text_align='center', text_color='red')
p_stdp.add_layout(ltp_label)
p_stdp.add_layout(ltd_label)

hover_stdp = HoverTool(tooltips=[('Δt', '@x{0.0} ms'), ('Δw', '@y{0.0000}')])
p_stdp.add_tools(hover_stdp)
p_stdp.legend.location = 'top_right'

show(p_stdp)

print("\n📊 STDP Learning Window")
print("   Green: Potentiation (LTP) when pre-spike leads post-spike")
print("   Red: Depression (LTD) when post-spike leads pre-spike")
print("   Hover to see exact weight changes")

In [ ]:
# Short-term plasticity simulation - FIXED API

# Depressing synapse (common in cortical excitatory connections)
stp_dep = ShortTermPlasticity(U=0.5, tau_rec=800.0, tau_facil=0.0, dt=1.0)

# Facilitating synapse (common in hippocampal connections)
stp_fac = ShortTermPlasticity(U=0.15, tau_rec=100.0, tau_facil=1000.0, dt=1.0)

# Spike train at different frequencies
frequencies = [5, 10, 20, 40]  # Hz
duration_stp = 1000  # ms
dt_stp = 1.0  # ms
batch_size = 1

results_dep = {}
results_fac = {}

for freq in frequencies:
    # Generate spike times
    isi = 1000 / freq  # ms
    spike_times = np.arange(0, duration_stp, isi)
    n_spikes = len(spike_times)
    
    # Reset synapses
    stp_dep_test = ShortTermPlasticity(U=0.5, tau_rec=800.0, tau_facil=0.0, dt=dt_stp)
    stp_fac_test = ShortTermPlasticity(U=0.15, tau_rec=100.0, tau_facil=1000.0, dt=dt_stp)
    
    # Initialize state
    if stp_dep_test.x is None:
        stp_dep_test.x = torch.ones(batch_size, 1)
        stp_dep_test.u = torch.ones(batch_size, 1) * stp_dep_test.U
    if stp_fac_test.x is None:
        stp_fac_test.x = torch.ones(batch_size, 1)
        stp_fac_test.u = torch.ones(batch_size, 1) * stp_fac_test.U
    
    responses_dep = []
    responses_fac = []
    
    spike_idx = 0
    for t_ms in range(int(duration_stp)):
        # Check if spike occurs at this time
        spike = 0
        if spike_idx < len(spike_times) and abs(t_ms - spike_times[spike_idx]) < 0.5:
            spike = 1
            spike_idx += 1
        
        if spike:
            # Record responses
            responses_dep.append((stp_dep_test.u * stp_dep_test.x).item())
            responses_fac.append((stp_fac_test.u * stp_fac_test.x).item())
            
            # Update on spike
            stp_dep_test.u = stp_dep_test.u + stp_dep_test.U * (1 - stp_dep_test.u)
            stp_dep_test.x = stp_dep_test.x - stp_dep_test.u * stp_dep_test.x
            
            stp_fac_test.u = stp_fac_test.u + stp_fac_test.U * (1 - stp_fac_test.u)
            stp_fac_test.x = stp_fac_test.x - stp_fac_test.u * stp_fac_test.x
        
        # Decay
        stp_dep_test.u = stp_dep_test.u * np.exp(-dt_stp / 1e-6)  # Fast decay
        stp_dep_test.x = stp_dep_test.x + (1 - stp_dep_test.x) * (1 - np.exp(-dt_stp / stp_dep_test.tau_rec))
        
        stp_fac_test.u = stp_fac_test.u + (stp_fac_test.U - stp_fac_test.u) * (1 - np.exp(-dt_stp / stp_fac_test.tau_facil))
        stp_fac_test.x = stp_fac_test.x + (1 - stp_fac_test.x) * (1 - np.exp(-dt_stp / stp_fac_test.tau_rec))
    
    results_dep[freq] = (np.arange(len(responses_dep)), responses_dep)
    results_fac[freq] = (np.arange(len(responses_fac)), responses_fac)

print("✓ Short-term plasticity simulated at multiple frequencies")

In [ ]:
# Interactive STP visualization

from bokeh.palettes import Category10
colors = Category10[4]

# Depressing synapse
p_dep = figure(
    title='Short-Term Depression',
    width=550,
    height=400,
    x_axis_label='Spike Number',
    y_axis_label='Synaptic Response (normalized)'
)

for i, freq in enumerate(frequencies):
    spike_nums, responses = results_dep[freq]
    if len(responses) > 0:
        p_dep.line(spike_nums + 1, responses, line_width=2, color=colors[i], 
                   legend_label=f'{freq} Hz', alpha=0.8)
        p_dep.circle(spike_nums + 1, responses, size=6, color=colors[i], alpha=0.6)

hover_dep = HoverTool(tooltips=[('Spike #', '@x'), ('Response', '@y{0.000}')])
p_dep.add_tools(hover_dep)
p_dep.legend.location = 'top_right'

# Facilitating synapse
p_fac = figure(
    title='Short-Term Facilitation',
    width=550,
    height=400,
    x_axis_label='Spike Number',
    y_axis_label='Synaptic Response (normalized)'
)

for i, freq in enumerate(frequencies):
    spike_nums, responses = results_fac[freq]
    if len(responses) > 0:
        p_fac.line(spike_nums + 1, responses, line_width=2, color=colors[i], 
                   legend_label=f'{freq} Hz', alpha=0.8)
        p_fac.circle(spike_nums + 1, responses, size=6, color=colors[i], alpha=0.6)

hover_fac = HoverTool(tooltips=[('Spike #', '@x'), ('Response', '@y{0.000}')])
p_fac.add_tools(hover_fac)
p_fac.legend.location = 'bottom_right'

# Combine
grid_stp = gridplot([[p_dep, p_fac]])
show(grid_stp)

print("\n📊 Short-Term Plasticity")
print("   Left: Depression - response decreases with repeated stimulation")
print("   Right: Facilitation - response increases with high-frequency input")

---

## Part 4: Metabolic Constraints

### ATP Dynamics and Energy Budgets

Neural computation is metabolically expensive. The brain consumes ~20% of the body's energy while comprising only ~2% of body weight. **ATP** (adenosine triphosphate) is the energy currency.

**Key findings** (Attwell & Laughlin, 2001):
- Action potentials: ~10⁸ ATP molecules each
- Synaptic transmission: ~10⁷ ATP molecules per vesicle
- Resting potential maintenance: ~50% of total budget

### Metabolic Realism in Foundation Models

Can foundation models operate within biological energy budgets?

In [ ]:
# ATP dynamics simulation - FIXED API
atp_model = ATPDynamics(
    ATP_rest=2.5,        # mM (baseline)
    ADP_rest=0.01,       # mM
    glucose=5.0,         # mM
    production_rate=1.0, # Relative rate
    dt=1.0               # ms
)

# Initialize state
batch_size = 1
if atp_model.ATP is None:
    atp_model.ATP = torch.ones(batch_size, 1) * atp_model.ATP_rest
    atp_model.ADP = torch.ones(batch_size, 1) * atp_model.ADP_rest
    atp_model.Pi = torch.ones(batch_size, 1) * 1.0

# Simulation parameters
duration_atp = 10000  # ms (10 seconds)
dt_atp = 1.0  # ms
n_steps_atp = int(duration_atp / dt_atp)

# Create activity pattern
time_atp = np.arange(n_steps_atp) * dt_atp / 1000  # seconds
baseline_rate = 5.0  # Hz
spike_rate = baseline_rate + 10.0 * np.sin(2 * np.pi * 0.5 * time_atp)  # Oscillating
spike_rate = np.maximum(0, spike_rate)

# Add burst epochs
burst_times = [2, 6]  # seconds
for burst_t in burst_times:
    burst_mask = (time_atp >= burst_t) & (time_atp < burst_t + 0.5)
    spike_rate[burst_mask] += 30.0

# Simulate
atp_trace = []
spike_cost = 0.002  # mM per spike (simplified)

for i in range(n_steps_atp):
    # Poisson spike generation
    spike_prob = spike_rate[i] * dt_atp / 1000
    spike = 1 if np.random.rand() < spike_prob else 0
    
    # Update ATP (simplified dynamics)
    production = atp_model.glycolysis_rate(torch.tensor([[atp_model.glucose]]), atp_model.ATP)
    consumption = spike * spike_cost
    
    atp_model.ATP += (production.item() - consumption) * dt_atp
    atp_model.ATP = torch.clamp(atp_model.ATP, min=0.1)
    
    atp_trace.append(atp_model.ATP.item())

atp_trace = np.array(atp_trace)

print(f"✓ ATP dynamics simulated for {duration_atp/1000} seconds")
print(f"  Mean ATP: {atp_trace.mean():.3f} mM")
print(f"  Min ATP: {atp_trace.min():.3f} mM")
print(f"  ATP depletion: {(1 - atp_trace.min() / atp_model.ATP_rest) * 100:.1f}%")

In [ ]:
# Interactive ATP dynamics visualization

# Panel 1: Spike rate (input)
p_rate = figure(
    title='Neural Activity Pattern',
    width=900,
    height=200,
    x_axis_label='Time (s)',
    y_axis_label='Spike Rate (Hz)'
)
p_rate.line(time_atp, spike_rate, line_width=2, color='black')
for burst_t in burst_times:
    burst_span = Span(location=burst_t, dimension='height', 
                     line_color='red', line_dash='dotted', line_width=2, line_alpha=0.3)
    p_rate.add_layout(burst_span)
hover_rate = HoverTool(tooltips=[('Time', '@x{0.0} s'), ('Rate', '@y{0.0} Hz')])
p_rate.add_tools(hover_rate)

# Panel 2: ATP concentration
p_atp = figure(
    title='ATP Dynamics',
    width=900,
    height=300,
    x_axis_label='Time (s)',
    y_axis_label='[ATP] (mM)',
    x_range=p_rate.x_range
)
p_atp.line(time_atp, atp_trace, line_width=2, color='blue')
baseline_atp = Span(location=atp_model.ATP_rest, dimension='width', 
                    line_color='red', line_dash='dashed', line_width=2)
p_atp.add_layout(baseline_atp)
baseline_label = Label(x=0.5, y=atp_model.ATP_rest + 0.05, text='Baseline', 
                      text_font_size='10pt', text_color='red')
p_atp.add_layout(baseline_label)
hover_atp = HoverTool(tooltips=[('Time', '@x{0.0} s'), ('[ATP]', '@y{0.000} mM')])
p_atp.add_tools(hover_atp)

# Combine
grid_atp = gridplot([[p_rate], [p_atp]])
show(grid_atp)

print("\n📊 ATP Dynamics Visualization")
print("   Top: Neural firing rate with burst epochs")
print("   Bottom: ATP concentration (depletes during high activity)")

---

## Conclusion

This notebook demonstrated:

1. **Ion Channel Dynamics**: Hodgkin-Huxley formalism for realistic action potentials
2. **Multi-Compartment Neurons**: Cable theory and dendritic integration
3. **Synaptic Plasticity**: STDP and short-term plasticity mechanisms
4. **Metabolic Constraints**: ATP dynamics and energy budgets

**Key Takeaways:**

- Biophysical models bridge **molecular** (channels) to **network** (circuits) scales
- Energy constraints are a powerful lens for evaluating AI representations
- Plasticity rules like STDP emerge from biological constraints, not arbitrary design
- Foundation models often violate biological constraints → opportunities for improvement

**Next Steps:**

- Explore more complex neuron morphologies
- Test foundation models against biological constraints
- Implement homeostatic plasticity mechanisms

---

### References

1. Hodgkin, A. L., & Huxley, A. F. (1952). A quantitative description of membrane current and its application to conduction and excitation in nerve. *The Journal of Physiology*, 117(4), 500.

2. Rall, W. (1959). Branching dendritic trees and motoneuron membrane resistivity. *Experimental Neurology*, 1(5), 491-527.

3. Bi, G. Q., & Poo, M. M. (1998). Synaptic modifications in cultured hippocampal neurons: dependence on spike timing, synaptic strength, and postsynaptic cell type. *Journal of Neuroscience*, 18(24), 10464-10472.

4. Attwell, D., & Laughlin, S. B. (2001). An energy budget for signaling in the grey matter of the brain. *Journal of Cerebral Blood Flow & Metabolism*, 21(10), 1133-1145.

---

*Created with NeuroS-MechInt • Advanced Mechanistic Interpretability for Neuroscience Foundation Models*